# Data Wrangling: Join, Combine and Reshape
## DAT540 Introduction to Data Science
## University of Stavanger
### L11.1

- **Hierarchical Indexing**
- enables multiple (two or more) index levels on an axis
- Creates *MultiIndex* datastruct that holds multiple indices on an axis
- With a hierarchical index object, *partial* indexing is possible enabling concise ways to select subset of data
- It also allows selection from an "inner" level
- Hierarchical indexing plays an important role in reshaping data and group-based operations
- A Series with MultiIndex can be rearranged into a DataFrame using the *unstack* instance method
- Similarly a DataFrame can be converted into a MultiIndex series using the *stack* instance method that will transform the column labels as inner row indices
- DataFrames allows MultiIndex on either axis
- Hierarchical levels can have names as strings or any python object
- MultiIndex can be created and reused using its *from_arrays* method

- *Indexing with a DataFrame's columns*
- One or more columns of a DataFrame can be used to index its rows. Additionally, the rows can also be used to index the columns
- The *set_index* instance method can be used to index rows based on columns
- By default the columns are removed from the DataFrame and can be changed by specifying the *drop* argument as False
- *reset_index* instance method moves hierarchical index levels into columns

In [ ]:
import numpy as np
import pandas as pd

# Get the adult dataset from UCI Machine Learning repo
adult = pd.read_csv('./data/adult_sanitized.csv')

In [ ]:
# sort the adult dataset values by 'Native-Country', 'Sex'
adult.sort_values(by=['Native-Country', 'Sex'], inplace=True)
# strip any whitespaces for the column values
adult['Native-Country'] = adult['Native-Country'].map(lambda x: x.strip())
adult['Sex'] = adult['Sex'].map(lambda x: x.strip())

# set the index of the adult data set using set_index to 'Native-Country', 'Sex'. 
# Store the resulting dataset into a variable adult_mi
adult_mi = adult.set_index(['Native-Country', 'Sex'])
# Display the adult_mi head(3)
adult_mi.head(3)


In [ ]:
# Use loc to retrive all rows from adult_mi with multiindex ['United-States', 'Female']
adult_mi.loc[('United-States', 'Female')].head(3)


In [ ]:
# Create a Series with MultiIndex 
# index=[['a', 'a', 'a', 'b', 'b', 'c', 'c', 'd', 'd'], [1,2,3,1,3,1,2,2,3]]
# Gen a sequence of 9 randn
data = pd.Series(np.random.randn(9),
                index=[['a', 'a', 'a', 'b', 'b', 'c', 'c', 'd', 'd'], [1,2,3,1,3,1,2,2,3]])
data

In [ ]:
# partial indexing
data['b']

In [ ]:
# we can rearrange data to a DataFrame using unstack
# Since the original series has two levels 0 and 1, we can choose the level to unstack on (default innermost)
data.unstack(0)


In [ ]:
# The unstacked DataFrame can be transformed into a stacked series using the stack method
data.unstack().stack()


- *Reordering and Sorting Levels*
- The *swaplevel* instance method takes two level numbers or names and returns a new object with the levels interchanged
- *sort_index* instance method sorts the data using only the values in a single level

In [ ]:
# swap 'Native-Country', 'Sex' in adult_mi data set using instance method swaplevel
adult_mi.head(2)

In [ ]:
# In case adult_mi Native-Country will become the innermost index
adult_mi.swaplevel('Native-Country', 'Sex').head(2)

- **Combining and Merging Datasets**
- It is often necessary to join, combine or merge datasets from different repositories
- Data contained in pandas objects can be combined together in multiple ways
  - *pandas.merge* top-level method connects rows in DataFrames based on one or more keys. Similar to join operations
  - *pandas.concat* top-level method concatenates or "stacks" together objects along an axis
  - *combine_first* instance method enables splicing together with overlapping data to fill in missing values in one object with values from another
- *Database-Style DataFrame Joins*
- *Merge* or *Join* operations combine datasets by linking rows using one or more keys using *pandas.merge*
- The *on* argument specifies the column(s) to be used for performing a join
- Without the on argument, merge uses overlapping column names as keys
- If the column names are different in each object, they can be specified separately using *left_on* and *right_on* arguments for the left and right dataframes respectively
- By default merge does an *inner join*
- The *how* argument allows 'left', 'right' and 'outer' joins 
- To merge with multiple keys, a list of column names are passed to the *on* argument
- *suffixes=()* argument takes a tuple and adds suffixes to overlapping columns when performing merge

In [ ]:
np.random.seed(1234)
# Generate two data sets with random values.
df1 = pd.DataFrame({'key': ['b', 'b', 'a',  'c', 'a', 'a', 'd'],
                   'data1': np.random.randint(0, 10, 7)})
df2 = pd.DataFrame({'key': ['a', 'b', 'd'],
                   'data2': np.random.randint(0, 20, 3)})
print('df1:\n', df1)
print('df2:\n', df2)

In [ ]:
# Perfrom a default inner join using pd.merge on df1 and df2.
# pandas automatically chooses the overlapping columns as the keys.
# For performing merge on specific columns that overlap, it can be defined using argument "on". 
# Could be a single label or a list of overlapping labels
pd.merge(df1, df2) # inner merge: use only the key combinations found in both tables

In [ ]:
# Change the column names for key as key1 and key2 for df1 and df2 respectively
df1.columns = ['key1', 'data1']
df2.columns = ['key2', 'data2']
print('df1:\n', df1)
print('df2:\n', df2)

In [ ]:
# Try perfroming a simple/default merge on them now using pd.merge

try:
  pd.merge(df1, df2)
except:
  print('No common columns to perform merge on.')

In [ ]:
# add a new column "data" to df1 and df2 with random values (len=7 and len=3 respectively)
df1['data'] = np.random.randn(7)
df2['data'] = np.random.randn(3)

print('df1:\n', df1)
print('df2:\n', df2)

# Perform merge on df1 and df2 with "left_on" and "right_on" argument specifying the key columns for each DataFrame
# Also specify the type of join as outer using the *how* argument
# Additionally use "suffixes" argument to specify a string to append to overlapping names in the left and right DFs
pd.merge(df1, df2, left_on='key1', right_on='key2', how='outer', suffixes=('_l', '_r'))

- Merge function arguments

<img src='images/merge_args.png'>

- *Merging on Index*
- Merging can also be performed on row indices (hierarchical and non-hierarchical)
- The *left_index=True* or *right_index=True* (or both) arguments indicates that the index should be used as the merge key
- Alternatively DataFrame also provides the *join* instance method to perform merge operation on indices
- The *join* method takes in similar arguments like merge
- Index-on-index merging using join can also accept a list of DataFrames to perform multiple joins with multiple DataFrames

In [ ]:
# Transform df1 and df2 with set_index, to include key1 and key2 as their respective row indices 
df1.set_index('key1', inplace=True)
df2.set_index('key2', inplace=True)
print('df1:\n', df1)
print('df2:\n', df2)

In [ ]:
# Merge df1 and df2 using their indices.
# Use "left_index", "right_index" or both (when the index names are different)

pd.merge(df1, df2, left_index=True, right_index=True)

In [ ]:
# Perform merge on indices using "join" instance method
df1.join(df2, lsuffix='_l', rsuffix='_r', how='outer')